# CHƯƠNG 3 – PHÂN TÍCH VÀ TRỰC QUAN HÓA DỮ LIỆU DU LỊCH VIỆT NAM

### Nội dung
- **3.1 Histogram** – Phân phối doanh thu địa phương
- **3.2 Box Plot** – Độ phân tán và giá trị ngoại lai
- **3.3 Stacked Bar** – Cơ cấu khách trong nước và quốc tế
- **3.4 Line Chart** – Xu hướng doanh thu
- **3.5 Line Chart** – Xu hướng lượng khách
- **3.6 Bar Chart** – Top 10 tỉnh/thành năm 2023
- **3.7 Scatter Plot** – Mối quan hệ lượng khách và doanh thu
- **3.8 Correlation Heatmap** – Tương quan giữa các chỉ tiêu
- **3.9 Animated Bar Chart** – Thay đổi thứ hạng địa phương
- **3.10 Tổng hợp các insight chính**

> Dữ liệu địa phương được phân biệt rõ giữa `Cả nước`, `Vùng` và `Tỉnh/thành`. Các phân tích xếp hạng/phân phối địa phương chỉ sử dụng cấp `Tỉnh/thành` để tránh trộn cấp dữ liệu.


In [1]:
# =========================================================
# THƯ VIỆN, ĐƯỜNG DẪN VÀ DỮ LIỆU DÙNG CHUNG
# =========================================================

import sys
from pathlib import Path
from itertools import combinations

import pandas as pd
import plotly.express as px
import plotly.io as pio

# Hiển thị Plotly trực tiếp trong VS Code Notebook
pio.renderers.default = "plotly_mimetype"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root():
    """Tìm thư mục project có data/processed."""
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path(sys.prefix).parent
    ]

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate in checked:
            continue
        checked.add(candidate)

        file_nam = candidate / "data" / "processed" / "du_lieu_du_lich_nam_clean.csv"
        file_dia = candidate / "data" / "processed" / "du_lieu_du_lich_dia_phuong_clean.csv"

        if file_nam.exists() and file_dia.exists():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy thư mục data/processed chứa hai file clean. "
        "Hãy mở notebook trong project TQH-DU-LIEU-NHOM6."
    )


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

PATH_NAM = DATA_PATH / "du_lieu_du_lich_nam_clean.csv"
PATH_DIA = DATA_PATH / "du_lieu_du_lich_dia_phuong_clean.csv"

df_nam = pd.read_csv(PATH_NAM, encoding="utf-8-sig")
df_dia_phuong = pd.read_csv(PATH_DIA, encoding="utf-8-sig")

print("Project:", ROOT)
print("Dữ liệu theo năm:", df_nam.shape)
print("Dữ liệu địa phương:", df_dia_phuong.shape)
print("Giai đoạn:", df_nam["Năm"].min(), "-", df_nam["Năm"].max())


Project: D:\DEEP\TQH-DU-LIEU-NHOM6
Dữ liệu theo năm: (10, 13)
Dữ liệu địa phương: (700, 5)
Giai đoạn: 2015 - 2024


In [2]:
# =========================================================
# KIỂM TRA NHANH DỮ LIỆU ĐẦU VÀO
# =========================================================

missing_nam = df_nam.isna().sum()
missing_nam = missing_nam[missing_nam > 0]

print("CÁC CỘT CÒN GIÁ TRỊ THIẾU")
display(missing_nam.to_frame("Số giá trị thiếu"))

print("\nCẤP DỮ LIỆU ĐỊA PHƯƠNG")
display(
    df_dia_phuong["Cấp dữ liệu"]
    .value_counts()
    .to_frame("Số quan sát")
)


CÁC CỘT CÒN GIÁ TRỊ THIẾU


,Số giá trị thiếu
Khách nghỉ qua đêm (Nghìn lượt),6
Khách trong ngày (Nghìn lượt),6



CẤP DỮ LIỆU ĐỊA PHƯƠNG


,Số quan sát
Cấp dữ liệu,
Tỉnh/thành,630
Vùng,60
Cả nước,10


### Nhận xét dữ liệu đầu vào

- Hai biến `Khách nghỉ qua đêm` và `Khách trong ngày` còn thiếu dữ liệu nên không được chọn làm biến chính cho các biểu đồ quan hệ.
- Các chỉ tiêu sử dụng cho Scatter Plot và Heatmap có dữ liệu đầy đủ trong giai đoạn 2015–2024.
- Khi phân tích địa phương, chỉ sử dụng cấp `Tỉnh/thành` để tránh trộn với dữ liệu `Vùng` và `Cả nước`.


## 3.1. Histogram – Phân phối doanh thu du lịch lữ hành

**Mục tiêu:** Quan sát hình dạng phân phối doanh thu du lịch lữ hành của các tỉnh/thành trong giai đoạn nghiên cứu.


In [4]:
# =========================================================
# BIỂU ĐỒ 1 - HISTOGRAM
# =========================================================

hist_df = df_dia_phuong[
    df_dia_phuong["Cấp dữ liệu"] == "Tỉnh/thành"
].dropna(
    subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"]
).copy()

print("Số quan sát sử dụng:", len(hist_df))

fig1 = px.histogram(
    hist_df,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    nbins=30,
    marginal="box",
    title="Phân phối doanh thu du lịch lữ hành của các tỉnh/thành",
    labels={
        "Doanh thu du lịch lữ hành (Tỷ đồng)": "Doanh thu (Tỷ đồng)"
    }
)

fig1.update_layout(
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Số quan sát",
    bargap=0.05,
    template="plotly_white"
)

file_fig1 = FIGURES / "01_doanh_thu_dia_phuong.png"
fig1.write_image(file_fig1, scale=2)

fig1.show()
print(f"Đã lưu: {file_fig1.resolve()}")

Số quan sát sử dụng: 618


Đã lưu: D:\DEEP\TQH-DU-LIEU-NHOM6\outputs\figures\01_doanh_thu_dia_phuong.png


### Nhận xét Biểu đồ 1

- Phần lớn các quan sát tập trung ở mức doanh thu thấp, trong khi chỉ một số ít có doanh thu rất cao.
- Phân phối **lệch phải rõ rệt**, với phần đuôi kéo dài về phía các giá trị doanh thu lớn.
- Điều này cho thấy doanh thu du lịch lữ hành có sự chênh lệch lớn giữa các tỉnh/thành.


## 3.2. Box Plot – Độ phân tán và giá trị ngoại lai

**Mục tiêu:** Quan sát trung vị, độ phân tán và các giá trị doanh thu cao vượt trội giữa các tỉnh/thành.


In [5]:
# =========================================================
# BIỂU ĐỒ 2 - BOX PLOT
# =========================================================

box_df = df_dia_phuong[
    df_dia_phuong["Cấp dữ liệu"] == "Tỉnh/thành"
].dropna(
    subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"]
).copy()

fig2 = px.box(
    box_df,
    y="Doanh thu du lịch lữ hành (Tỷ đồng)",
    points="outliers",
    hover_data=["Địa phương", "Năm"],
    title="Box Plot doanh thu du lịch lữ hành của các tỉnh/thành",
    labels={
        "Doanh thu du lịch lữ hành (Tỷ đồng)": "Doanh thu (Tỷ đồng)"
    }
)

fig2.update_layout(
    yaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    template="plotly_white"
)

file_fig2 = FIGURES / "02_doanh_thu_dia_phuong.png"
fig2.write_image(file_fig2, scale=2)

fig2.show()
print(f"Đã lưu: {file_fig2.resolve()}")


Đã lưu: D:\DEEP\TQH-DU-LIEU-NHOM6\outputs\figures\02_doanh_thu_dia_phuong.png


### Nhận xét Biểu đồ 2

- Doanh thu giữa các tỉnh/thành có mức độ phân tán lớn.
- Biểu đồ xuất hiện nhiều **giá trị ngoại lai ở phía doanh thu cao**.
- Kết quả phù hợp với Histogram và tiếp tục cho thấy phân phối doanh thu lệch phải mạnh.


## 3.3. Pie Chart – Cơ cấu khách trong nước và quốc tế năm 2024

**Mục tiêu:** Phân tích tỷ trọng khách trong nước và khách quốc tế tại các cơ sở lưu trú trong năm 2024.

In [7]:
# =========================================================
# BIỂU ĐỒ 3 - STACKED BAR CHART
# =========================================================

stack_cols = [
    "Khách trong nước - cơ sở lưu trú (Nghìn lượt)",
    "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)"
]

stack_df = df_nam[["Năm"] + stack_cols].copy()

stack_long = stack_df.melt(
    id_vars="Năm",
    value_vars=stack_cols,
    var_name="Loại khách",
    value_name="Lượng khách (Nghìn lượt)"
)

stack_long["Loại khách"] = stack_long["Loại khách"].replace({
    "Khách trong nước - cơ sở lưu trú (Nghìn lượt)": "Khách trong nước",
    "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)": "Khách quốc tế"
})

fig3 = px.bar(
    stack_long,
    x="Năm",
    y="Lượng khách (Nghìn lượt)",
    color="Loại khách",
    barmode="stack",
    title="Cơ cấu khách trong nước và quốc tế tại cơ sở lưu trú",
    labels={
        "Năm": "Năm",
        "Lượng khách (Nghìn lượt)": "Lượng khách (Nghìn lượt)",
        "Loại khách": "Loại khách"
    }
)

fig3.update_layout(
    xaxis_title="Năm",
    yaxis_title="Lượng khách (Nghìn lượt)",
    legend_title="Loại khách",
    template="plotly_white"
)

fig3.update_xaxes(dtick=1)

file_fig3 = FIGURES / "03_co_cau_khach.png"
fig3.write_image(file_fig3, scale=2)

fig3.show()
print(f"Đã lưu: {file_fig3.resolve()}")


Đã lưu: D:\DEEP\TQH-DU-LIEU-NHOM6\outputs\figures\03_co_cau_khach.png


### Nhận xét Biểu đồ 3

- Khách trong nước chiếm phần lớn tổng lượng khách lưu trú trong toàn bộ giai đoạn 2015–2024.
- Giai đoạn 2020–2021, cả hai nhóm đều giảm mạnh, đặc biệt là khách quốc tế.
- Từ năm 2022, lượng khách phục hồi rõ rệt; thị trường nội địa đóng vai trò lớn trong quá trình phục hồi.


## 3.4. Line Chart – Xu hướng doanh thu du lịch giai đoạn 2015–2024

**Câu hỏi:** Doanh thu cơ sở lưu trú và cơ sở lữ hành biến động như thế nào? Giai đoạn nào suy giảm mạnh và quá trình phục hồi diễn ra ra sao?


In [ ]:
# =========================================================
# BIỂU ĐỒ 4 - XU HƯỚNG DOANH THU
# =========================================================

revenue_data = df_nam[
    [
        "Năm",
        "Doanh thu cơ sở lưu trú (Tỷ đồng)",
        "Doanh thu cơ sở lữ hành (Tỷ đồng)"
    ]
].copy()

revenue_long = revenue_data.melt(
    id_vars="Năm",
    value_vars=[
        "Doanh thu cơ sở lưu trú (Tỷ đồng)",
        "Doanh thu cơ sở lữ hành (Tỷ đồng)"
    ],
    var_name="Loại doanh thu",
    value_name="Doanh thu (Tỷ đồng)"
)

revenue_long["Loại doanh thu"] = revenue_long["Loại doanh thu"].replace({
    "Doanh thu cơ sở lưu trú (Tỷ đồng)": "Doanh thu lưu trú",
    "Doanh thu cơ sở lữ hành (Tỷ đồng)": "Doanh thu lữ hành"
})

fig4 = px.line(
    revenue_long,
    x="Năm",
    y="Doanh thu (Tỷ đồng)",
    color="Loại doanh thu",
    markers=True,
    title="Xu hướng doanh thu du lịch Việt Nam giai đoạn 2015–2024"
)

fig4.update_traces(
    line=dict(width=3),
    marker=dict(size=8),
    hovertemplate=(
        "Năm: %{x}<br>"
        "Doanh thu: %{y:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

fig4.update_layout(
    xaxis_title="Năm",
    yaxis_title="Doanh thu (Tỷ đồng)",
    legend_title="Loại doanh thu",
    template="plotly_white",
    hovermode="x unified"
)

fig4.update_xaxes(dtick=1)

file_fig4 = FIGURES / "04_xu_huong_doanh_thu.png"
fig4.write_image(file_fig4, scale=2)

fig4.show()
print(f"Đã lưu: {file_fig4.resolve()}")


### Nhận xét Biểu đồ 4

- Doanh thu lưu trú và lữ hành tăng trong giai đoạn 2015–2019, sau đó giảm mạnh vào 2020–2021.
- Năm 2021 là mức thấp nhất; doanh thu lữ hành giảm khoảng **80%**, mạnh hơn lưu trú khoảng **65%** so với năm 2019.
- Từ năm 2022, doanh thu phục hồi nhanh và năm 2024 đạt mức cao nhất trong giai đoạn.


## 3.5. Line Chart – Xu hướng lượng khách du lịch giai đoạn 2015–2024

**Câu hỏi:** Lượng khách tại cơ sở lưu trú và cơ sở lữ hành biến động như thế nào trong giai đoạn nghiên cứu?


In [ ]:
# =========================================================
# BIỂU ĐỒ 5 - XU HƯỚNG LƯỢNG KHÁCH
# =========================================================

visitor_data = df_nam[
    [
        "Năm",
        "Khách cơ sở lưu trú phục vụ (Nghìn lượt)",
        "Khách cơ sở lữ hành phục vụ (Nghìn lượt)"
    ]
].copy()

visitor_long = visitor_data.melt(
    id_vars="Năm",
    value_vars=[
        "Khách cơ sở lưu trú phục vụ (Nghìn lượt)",
        "Khách cơ sở lữ hành phục vụ (Nghìn lượt)"
    ],
    var_name="Loại khách",
    value_name="Lượng khách (Nghìn lượt)"
)

visitor_long["Loại khách"] = visitor_long["Loại khách"].replace({
    "Khách cơ sở lưu trú phục vụ (Nghìn lượt)": "Khách lưu trú",
    "Khách cơ sở lữ hành phục vụ (Nghìn lượt)": "Khách lữ hành"
})

fig5 = px.line(
    visitor_long,
    x="Năm",
    y="Lượng khách (Nghìn lượt)",
    color="Loại khách",
    markers=True,
    title="Xu hướng lượng khách du lịch Việt Nam giai đoạn 2015–2024"
)

fig5.update_traces(
    line=dict(width=3),
    marker=dict(size=8),
    hovertemplate=(
        "Năm: %{x}<br>"
        "Lượng khách: %{y:,.0f} nghìn lượt"
        "<extra></extra>"
    )
)

fig5.update_layout(
    xaxis_title="Năm",
    yaxis_title="Lượng khách (Nghìn lượt)",
    legend_title="Loại khách",
    template="plotly_white",
    hovermode="x unified"
)

fig5.update_xaxes(dtick=1)

file_fig5 = FIGURES / "05_xu_huong_luong_khach.png"
fig5.write_image(file_fig5, scale=2)

fig5.show()
print(f"Đã lưu: {file_fig5.resolve()}")


### Nhận xét Biểu đồ 5

- Lượng khách tăng trong giai đoạn 2015–2019 và giảm mạnh vào 2020–2021.
- Năm 2021 là mức thấp nhất; khách lữ hành giảm khoảng **81%**, mạnh hơn khách lưu trú khoảng **65%** so với năm 2019.
- Từ năm 2022, lượng khách phục hồi nhanh và năm 2024 đạt mức cao nhất trong giai đoạn.


## 3.6. Bar Chart – Top 10 tỉnh/thành theo doanh thu du lịch lữ hành năm 2023

**Câu hỏi:** Những tỉnh/thành nào dẫn đầu về doanh thu du lịch lữ hành trong năm 2023?

> Sử dụng năm 2023 vì đây là năm gần nhất có số liệu chính thức; số liệu 2024 trong bộ dữ liệu là sơ bộ.


In [ ]:
# =========================================================
# BIỂU ĐỒ 6 - TOP 10 TỈNH/THÀNH NĂM 2023
# =========================================================

top10_2023 = df_dia_phuong[
    (df_dia_phuong["Cấp dữ liệu"] == "Tỉnh/thành") &
    (df_dia_phuong["Năm"] == 2023)
].dropna(
    subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"]
).nlargest(
    10,
    "Doanh thu du lịch lữ hành (Tỷ đồng)"
).copy()

print("TOP 10 TỈNH/THÀNH NĂM 2023")
display(
    top10_2023[
        ["Địa phương", "Doanh thu du lịch lữ hành (Tỷ đồng)"]
    ].reset_index(drop=True)
)

top10_plot = top10_2023.sort_values(
    "Doanh thu du lịch lữ hành (Tỷ đồng)",
    ascending=True
)

fig6 = px.bar(
    top10_plot,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    text="Doanh thu du lịch lữ hành (Tỷ đồng)",
    title="Top 10 tỉnh/thành theo doanh thu du lịch lữ hành năm 2023"
)

fig6.update_traces(
    texttemplate="%{x:,.1f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Doanh thu: %{x:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

fig6.update_layout(
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Tỉnh/thành",
    template="plotly_white",
    showlegend=False
)

fig6.update_xaxes(
    type="linear",
    tickformat=","
)

file_fig6 = FIGURES / "06_top10_dia_phuong_2023.png"
fig6.write_image(file_fig6, scale=2)

fig6.show()
print(f"Đã lưu: {file_fig6.resolve()}")


### Nhận xét Biểu đồ 6

- **TP. Hồ Chí Minh** và **Hà Nội** dẫn đầu với lần lượt khoảng **25.580** và **20.687 tỷ đồng**, vượt xa các địa phương còn lại.
- **Đà Nẵng** đứng thứ ba với khoảng **4.579 tỷ đồng**, cho thấy khoảng cách lớn giữa hai địa phương dẫn đầu và nhóm tiếp theo.
- Từ vị trí thứ 5 trở đi, doanh thu đều dưới khoảng **1.100 tỷ đồng**, cho thấy doanh thu lữ hành tập trung mạnh ở nhóm dẫn đầu.


## 3.7. Scatter Plot – Mối quan hệ giữa lượng khách và doanh thu

**Mục tiêu:** Phân tích mối quan hệ giữa lượng khách cơ sở lưu trú phục vụ và doanh thu cơ sở lưu trú trong giai đoạn 2015–2024.


In [ ]:
# =========================================================
# BIỂU ĐỒ 7 - SCATTER PLOT
# =========================================================

col_khach = "Khách cơ sở lưu trú phục vụ (Nghìn lượt)"
col_doanh_thu = "Doanh thu cơ sở lưu trú (Tỷ đồng)"

scatter_data = df_nam[
    ["Năm", col_khach, col_doanh_thu]
].dropna()

pearson_r = scatter_data[col_khach].corr(
    scatter_data[col_doanh_thu]
)

print(f"Hệ số tương quan Pearson: r = {pearson_r:.3f}")

fig7 = px.scatter(
    scatter_data,
    x=col_khach,
    y=col_doanh_thu,
    text="Năm",
    title="Mối quan hệ giữa lượng khách và doanh thu lưu trú"
)

fig7.update_traces(
    marker=dict(size=11),
    textposition="top center",
    hovertemplate=(
        "Năm: %{text}<br>"
        "Khách: %{x:,.0f} nghìn lượt<br>"
        "Doanh thu: %{y:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

fig7.update_layout(
    xaxis_title="Khách cơ sở lưu trú phục vụ (Nghìn lượt)",
    yaxis_title="Doanh thu cơ sở lưu trú (Tỷ đồng)",
    template="plotly_white"
)

file_fig7 = FIGURES / "07_khach_doanh_thu.png"
fig7.write_image(file_fig7, scale=2)

fig7.show()
print(f"Đã lưu: {file_fig7.resolve()}")


### Nhận xét Biểu đồ 7

- Lượng khách và doanh thu cơ sở lưu trú có xu hướng tăng, giảm cùng chiều.
- Hệ số **Pearson r = 0.994** cho thấy tương quan tuyến tính dương rất mạnh giữa hai chỉ tiêu.
- Năm 2021 ở mức thấp nhất, trong khi năm 2024 đạt mức cao nhất về cả lượng khách và doanh thu.


## 3.8. Correlation Heatmap – Tương quan giữa các chỉ tiêu du lịch

**Mục tiêu:** So sánh mức độ tương quan giữa các chỉ tiêu doanh thu và lượng khách du lịch.


In [ ]:
# =========================================================
# BIỂU ĐỒ 8 - CORRELATION HEATMAP
# =========================================================

corr_cols = {
    "Doanh thu cơ sở lưu trú (Tỷ đồng)": "DT lưu trú",
    "Doanh thu cơ sở lữ hành (Tỷ đồng)": "DT lữ hành",
    "Khách cơ sở lưu trú phục vụ (Nghìn lượt)": "Khách lưu trú",
    "Khách trong nước - cơ sở lưu trú (Nghìn lượt)": "Khách nội địa",
    "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)": "Khách quốc tế",
    "Khách cơ sở lữ hành phục vụ (Nghìn lượt)": "Khách lữ hành"
}

corr_data = df_nam[list(corr_cols.keys())].rename(columns=corr_cols)
corr_matrix = corr_data.corr(method="pearson")

fig8 = px.imshow(
    corr_matrix,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Ma trận tương quan giữa các chỉ tiêu du lịch"
)

fig8.update_layout(
    width=900,
    height=650,
    xaxis_title="Chỉ tiêu",
    yaxis_title="Chỉ tiêu"
)

pairs = [
    (a, b, corr_matrix.loc[a, b])
    for a, b in combinations(corr_matrix.columns, 2)
]

strongest = max(pairs, key=lambda x: abs(x[2]))
lowest = min(pairs, key=lambda x: abs(x[2]))

print(f"Mạnh nhất: {strongest[0]} ↔ {strongest[1]} (r = {strongest[2]:.3f})")
print(f"Thấp nhất: {lowest[0]} ↔ {lowest[1]} (r = {lowest[2]:.3f})")

file_fig8 = FIGURES / "08_heatmap.png"
fig8.write_image(file_fig8, scale=2)

fig8.show()
print(f"Đã lưu: {file_fig8.resolve()}")


### Nhận xét Biểu đồ 8

- Các chỉ tiêu được phân tích đều có tương quan dương mạnh.
- Mạnh nhất là **doanh thu lưu trú và khách lưu trú** với **r = 0.994**.
- Thấp nhất là **khách nội địa và khách quốc tế** với **r = 0.816**, nhưng vẫn cho thấy tương quan dương mạnh.


## 3.9. Animated Bar Chart – Thay đổi thứ hạng doanh thu địa phương

**Mục tiêu:** Theo dõi sự thay đổi Top 10 tỉnh/thành có doanh thu du lịch lữ hành cao nhất trong giai đoạn 2015–2024.

> Năm 2024 là số liệu sơ bộ.


In [ ]:
# =========================================================
# BIỂU ĐỒ 9 - ANIMATED BAR CHART
# =========================================================

df_tinh = df_dia_phuong[
    df_dia_phuong["Cấp dữ liệu"] == "Tỉnh/thành"
].dropna(
    subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"]
).copy()

print("Số quan sát hợp lệ:", len(df_tinh))
print("Số tỉnh/thành:", df_tinh["Địa phương"].nunique())
print("Giai đoạn:", df_tinh["Năm"].min(), "-", df_tinh["Năm"].max())

top10_year = (
    df_tinh
    .sort_values(
        ["Năm", "Doanh thu du lịch lữ hành (Tỷ đồng)"],
        ascending=[True, False]
    )
    .groupby("Năm")
    .head(10)
    .copy()
)

top10_year["Hạng"] = (
    top10_year
    .groupby("Năm")["Doanh thu du lịch lữ hành (Tỷ đồng)"]
    .rank(method="first", ascending=False)
    .astype(int)
)

top10_animation = top10_year.sort_values(
    ["Năm", "Doanh thu du lịch lữ hành (Tỷ đồng)"],
    ascending=[True, True]
)

max_revenue = top10_animation[
    "Doanh thu du lịch lữ hành (Tỷ đồng)"
].max()

fig9 = px.bar(
    top10_animation,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    animation_frame="Năm",
    animation_group="Địa phương",
    text="Doanh thu du lịch lữ hành (Tỷ đồng)",
    title="Top 10 tỉnh/thành theo doanh thu du lịch lữ hành qua các năm"
)

fig9.update_traces(
    texttemplate="%{x:,.1f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Doanh thu: %{x:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

fig9.update_layout(
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Tỉnh/thành",
    xaxis=dict(range=[0, max_revenue * 1.12]),
    showlegend=False,
    template="plotly_white"
)

file_animation = FIGURES / "09_animated_bar_top10_dia_phuong.html"
fig9.write_html(file_animation, include_plotlyjs=True)

fig9.show()
print(f"Đã lưu biểu đồ động: {file_animation.resolve()}")


# Ảnh tĩnh Top 10 năm 2024 để đưa vào báo cáo/slide
top10_2024 = (
    top10_year[top10_year["Năm"] == 2024]
    .sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)", ascending=True)
)

fig9_2024 = px.bar(
    top10_2024,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    text="Doanh thu du lịch lữ hành (Tỷ đồng)",
    title="Top 10 tỉnh/thành theo doanh thu du lịch lữ hành năm 2024 (sơ bộ)"
)

fig9_2024.update_traces(
    texttemplate="%{x:,.1f}",
    textposition="outside",
    cliponaxis=False
)

fig9_2024.update_layout(
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Tỉnh/thành",
    xaxis=dict(range=[0, max_revenue * 1.12]),
    showlegend=False,
    template="plotly_white"
)

file_fig9 = FIGURES / "09_top10_dia_phuong_2024.png"
fig9_2024.write_image(file_fig9, scale=2)

print(f"Đã lưu ảnh tĩnh: {file_fig9.resolve()}")


In [ ]:
# =========================================================
# SO SÁNH THỨ HẠNG TOP 10 NĂM 2015 VÀ 2024
# =========================================================

rank_2015 = (
    top10_year[top10_year["Năm"] == 2015]
    [["Địa phương", "Hạng"]]
    .rename(columns={"Hạng": "Hạng 2015"})
)

rank_2024 = (
    top10_year[top10_year["Năm"] == 2024]
    [["Địa phương", "Hạng"]]
    .rename(columns={"Hạng": "Hạng 2024"})
)

compare_rank = pd.merge(
    rank_2015,
    rank_2024,
    on="Địa phương",
    how="outer"
)

compare_rank = (
    compare_rank
    .sort_values(["Hạng 2024", "Hạng 2015"], na_position="last")
    .reset_index(drop=True)
)

print("SO SÁNH TOP 10 NĂM 2015 VÀ 2024")

display(
    compare_rank.style.format(
        {
            "Hạng 2015": "{:.0f}",
            "Hạng 2024": "{:.0f}"
        },
        na_rep="—"
    )
)


### Nhận xét Biểu đồ 9

- **TP. Hồ Chí Minh, Hà Nội và Đà Nẵng** giữ ba vị trí dẫn đầu ở cả năm 2015 và 2024.
- **Khánh Hòa** tăng từ hạng 7 lên hạng 4, cho thấy sự cải thiện rõ về thứ hạng.
- Năm 2024 có thêm **Hà Nam, Bình Định, Hải Phòng và Cần Thơ** vào Top 10; trong khi **Quảng Nam, Bà Rịa - Vũng Tàu, Quảng Bình và Huế** không còn trong nhóm này.
- Nhìn chung, nhóm dẫn đầu khá ổn định nhưng các vị trí còn lại có sự thay đổi theo thời gian.


## 3.10. Tổng hợp các insight chính

1. **Ngành du lịch thể hiện ba giai đoạn rõ rệt:** tăng trưởng trong 2015–2019, suy giảm mạnh trong 2020–2021 và phục hồi nhanh từ 2022. Đến **2024 (sơ bộ)**, các chỉ tiêu doanh thu và lượng khách chính đều đạt mức cao nhất của giai đoạn.

2. **Lữ hành biến động mạnh hơn lưu trú trong giai đoạn suy giảm.** So với năm 2019, năm 2021 doanh thu lữ hành giảm khoảng **80%** và khách lữ hành giảm khoảng **81%**, trong khi doanh thu và lượng khách lưu trú giảm khoảng **65%**.

3. **Khách trong nước là thành phần chủ đạo của thị trường lưu trú.** Stacked Bar cho thấy khách nội địa chiếm phần lớn tổng lượng khách trong toàn giai đoạn và đóng vai trò quan trọng trong quá trình phục hồi từ năm 2022; khách quốc tế có mức biến động mạnh hơn.

4. **Doanh thu lữ hành giữa các tỉnh/thành phân bố rất không đồng đều.** Histogram lệch phải và Box Plot có nhiều ngoại lai phía cao cho thấy phần lớn địa phương có doanh thu thấp hơn, trong khi một nhóm nhỏ địa phương có quy mô vượt trội.

5. **Doanh thu năm 2023 tập trung mạnh ở hai địa phương dẫn đầu.** TP. Hồ Chí Minh và Hà Nội đạt tổng khoảng **46.267 tỷ đồng**, tương đương khoảng **79% tổng doanh thu của nhóm Top 10 năm 2023**, trong khi Đà Nẵng đứng thứ ba nhưng có khoảng cách lớn.

6. **Lượng khách và doanh thu lưu trú có quan hệ rất chặt chẽ.** Scatter Plot cho hệ số **Pearson r = 0.994**. Heatmap cũng cho thấy các cặp chỉ tiêu được chọn đều có tương quan dương mạnh; cặp thấp nhất vẫn đạt khoảng **r = 0.816**. Tương quan mạnh không đồng nghĩa với quan hệ nhân quả.

7. **Thứ hạng địa phương vừa có tính ổn định vừa có sự dịch chuyển.** TP. Hồ Chí Minh, Hà Nội và Đà Nẵng giữ Top 3 từ 2015 đến 2024; Khánh Hòa tăng từ hạng 7 lên hạng 4, đồng thời Top 10 năm 2024 xuất hiện thêm Hà Nam, Bình Định, Hải Phòng và Cần Thơ.


## Kết luận

Các biểu đồ 3.1–3.9 bổ trợ cho nhau theo ba hướng chính: **phân phối**, **xu hướng** và **mối quan hệ/thứ hạng**. Kết quả cho thấy du lịch Việt Nam chịu cú giảm mạnh trong 2020–2021 nhưng phục hồi rõ rệt từ 2022, đồng thời doanh thu có mức độ tập trung cao ở một số địa phương và có quan hệ chặt với quy mô khách phục vụ.
